# Predictive Maintenance — End-to-end notebookThis notebook runs Exploratory Data Analysis (EDA), preprocessing, feature engineering, model training, evaluation, and saves the trained model for the `ai4i2020.csv` dataset.## InstructionsRun the cells from top to bottom. If required packages are not installed, run the installation cell.

In [ ]:
# Optional: install required packages (uncomment to run)# Note: running pip from inside the notebook may restart the kernel for compiled packages.# import sys# !{sys.executable} -m pip install -r '../requirements.txt'print('Skip installation if dependencies are already present.')

In [ ]:
import osimport pandas as pdimport numpy as npimport matplotlib.pyplot as pltimport seaborn as snsfrom sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_scorefrom sklearn.ensemble import RandomForestClassifierfrom sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, average_precision_score, precision_recall_curvefrom sklearn.preprocessing import StandardScaler, OneHotEncoderfrom sklearn.compose import ColumnTransformerfrom sklearn.pipeline import Pipelineimport joblibprint('Imports complete')

In [ ]:
# Load datasetdata_path = r'c:\Users\HP\OneDrive\Documents\Auto dataset\ai4i2020.csv'df = pd.read_csv(data_path)df.head()

## Quick EDA

In [ ]:
# Shape & typesprint('Shape:', df.shape)print('Dtypes:')print(df.dtypes)# Missing valuesprint('Missing values:')print(df.isna().sum())# Target distributionif 'Machine failure' in df.columns:    print('Machine failure counts:')    print(df['Machine failure'].value_counts())    print('Machine failure %:')    print(df['Machine failure'].value_counts(normalize=True) * 100)# Basic numeric descriptionprint('Numeric summary:')print(df.describe().T)

In [ ]:
num = df.select_dtypes(include=[np.number])corrs = num.corr()['Machine failure'].sort_values(ascending=False)print(corrs)  # Correlation with target (numeric features)if 'Machine failure' in df.columns:    plt.figure(figsize=(6,4))    sns.countplot(x='Machine failure', data=df)    plt.title('Machine failure distribution')    plt.show()

## Preprocessing and Feature Engineering

In [ ]:
df_proc = df.copy()# Drop identifiers that are not predictivefor c in ['UDI','Product ID']:    if c in df_proc.columns:        df_proc.drop(columns=c, inplace=True)# Create derived features# Interaction: speed * torqueif 'Rotational speed [rpm]' in df_proc.columns and 'Torque [Nm]' in df_proc.columns:    df_proc['speed_torque'] = df_proc['Rotational speed [rpm]'] * df_proc['Torque [Nm]']# Ratio: process / air tempif 'Process temperature [K]' in df_proc.columns and 'Air temperature [K]' in df_proc.columns:    df_proc['temp_ratio'] = df_proc['Process temperature [K]'] / df_proc['Air temperature [K]']# Tool wear bucketif 'Tool wear [min]' in df_proc.columns:    df_proc['tool_wear_bin'] = pd.cut(df_proc['Tool wear [min]'], bins=[-1,50,100,150,200,300], labels=['0-50','51-100','101-150','151-200','200+'])df_proc.head()

In [ ]:
# Define features and targettarget = 'Machine failure'if target not in df_proc.columns:    raise ValueError('Target column not found')# Select feature columns (exclude failure-mode flags if you'd like to predict them separately)exclude = ['TWF','HDF','PWF','OSF','RNF']  # these are failure-mode indicators present in the datasetfeatures = [c for c in df_proc.columns if c != target and c not in exclude]print('Using features:', features)X = df_proc[features]y = df_proc[target]# SplitX_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)print('Train size:', X_train.shape, 'Test size:', X_test.shape)

In [ ]:
# Preprocessing pipeline: numeric scaling + one-hot for 'Type' and tool_wear_binnumeric_features = X_train.select_dtypes(include=[np.number]).columns.tolist()categorical_features = [c for c in X_train.columns if c not in numeric_features]numeric_transformer = Pipeline(steps=[('scaler', StandardScaler())])categorical_transformer = Pipeline(steps=[('onehot', OneHotEncoder(handle_unknown='ignore'))])preprocessor = ColumnTransformer(transformers=[    ('num', numeric_transformer, numeric_features),    ('cat', categorical_transformer, categorical_features)])# Choose model: try LightGBM if available, otherwise RandomForest with class balancetry:    import lightgbm as lgb    model = lgb.LGBMClassifier(n_estimators=200, random_state=42)    print('Using LightGBM')except Exception as e:    model = RandomForestClassifier(n_estimators=200, class_weight='balanced', random_state=42)    print('LightGBM not available, using RandomForest')clf = Pipeline(steps=[('preprocessor', preprocessor), ('classifier', model)])# Fitclf.fit(X_train, y_train)print('Model trained')

## Evaluation

In [ ]:
# Predictions and metricsy_pred = clf.predict(X_test)y_proba = Nonetry:    y_proba = clf.predict_proba(X_test)[:,1]except Exception:    passprint('Classification report:')print(classification_report(y_test, y_pred, digits=4))print('Confusion matrix:')print(confusion_matrix(y_test, y_pred))if y_proba is not None:    print('ROC AUC:', roc_auc_score(y_test, y_proba))    print('PR AUC (average precision):', average_precision_score(y_test, y_proba))    precision, recall, _ = precision_recall_curve(y_test, y_proba)    plt.figure(figsize=(6,4))    plt.plot(recall, precision, label='Precision-Recall curve')    plt.xlabel('Recall')    plt.ylabel('Precision')    plt.title('Precision-Recall curve')    plt.legend()    plt.show()

In [ ]:
# Feature importance (sklearn or LightGBM access)try:    clf_steps = clf.named_steps    # Get transformed feature names    ohe = clf_steps['preprocessor'].named_transformers_['cat'].named_steps['onehot']    cat_names = ohe.get_feature_names_out(categorical_features) if hasattr(ohe, 'get_feature_names_out') else ohe.get_feature_names(categorical_features)    feature_names = list(numeric_features) + list(cat_names)    importances = None    if hasattr(clf_steps['classifier'], 'feature_importances_'):        importances = clf_steps['classifier'].feature_importances_    elif hasattr(clf_steps['classifier'], 'booster'):        importances = clf_steps['classifier'].booster_.feature_importance()    if importances is not None:        fi = pd.Series(importances, index=feature_names).sort_values(ascending=False).head(20)        print(fi)        fi.plot(kind='barh', figsize=(8,6))        plt.gca().invert_yaxis()        plt.show()except Exception as e:    print('Could not compute feature importances:', e)

## Save model and pipeline

In [ ]:
out_path = r'c:\Users\HP\OneDrive\Documents\Auto dataset\models'os.makedirs(out_path, exist_ok=True)model_file = os.path.join(out_path, 'pm_pipeline.joblib')joblib.dump(clf, model_file)print('Saved pipeline to', model_file)

## Next steps and notes- Consider time-window/sequence features per `Product ID` if sensor history is available.- Use cost-sensitive loss or tune probability threshold to prioritize recall (catch failures).- Add model explainability (SHAP) for production monitoring and root-cause analysis.- Schedule periodic retraining to handle concept drift.